## 공개용 코드 안내

FINAL2에서 생성한 Dev/Test permutation cache와 V3 예측 CSV를 재사용하는 후처리 실험입니다. **최고 Public Score 0.96127은 FINAL2의 Vote-Switch 제출 결과이며, FINAL3가 그보다 높은 점수를 냈다는 의미는 아닙니다.**


# FINAL3 — Vote-Switch Micro-Tuning (No Retraining / No New Inference)

현재 최고 Public Score:

```text
04_perm_vote_switch.csv = 0.96127
```

이 결과가 지금까지 가장 좋았으므로, 새로운 모델/학습/추론은 더 하지 않습니다.

이번 notebook은 FINAL2에서 이미 만든:

```text
dev_perm_cache.csv
test_perm_cache.csv
```

를 그대로 읽어서 **Vote-Switch 규칙만 미세 조정**합니다.

즉 실행 시간은 거의 CSV 후처리 수준입니다.

---

## 현재 성공한 규칙

FINAL2의 최고 제출은 대략:

```text
3개 permutation 중 2개 이상이 같은 alternative에 투표
AND
permutation 평균 확률에서 alternative가 기존 V3 답보다 0.08 이상 우세
AND
V3가 low-margin 문제
→ 기존 V3 답 변경
```

이 규칙으로 **35개 답만 변경**했고 Public Score가:

```text
0.95918 → 0.96127
```

로 상승했습니다.

따라서 이번에는 이 규칙 주변만 좁게 탐색합니다.

---

## 생성되는 주요 후보

```text
00_anchor_delta008.csv        # 기존 0.96127 재현용
01_looser_delta006.csv        # 조금 더 많이 변경
02_tighter_delta010.csv       # 조금 더 보수적
03_tighter_delta012.csv
04_unanimous_3of3.csv         # 3개 permutation 전부 같은 답일 때만
05_unanimous_delta005.csv
06_dev_best_safe.csv          # Dev 최적 + Test 변경 10~60개 제한
07_dev_best_ultra_safe.csv    # Dev 최적 + Test 변경 10~40개 제한
```

추가로 모든 조합을 `all_rule_candidates.csv`에 기록합니다.

### 제출 추천

현재 `0.96127`이 이미 강하므로 대규모 변경은 피합니다.

제출 여유가 적다면:

1. `02_tighter_delta010.csv`
2. `01_looser_delta006.csv`
3. `04_unanimous_3of3.csv`

순으로 보는 것을 권장합니다.

`00_anchor_delta008.csv`는 기존 최고 제출 재현용이라 다시 제출할 필요 없습니다.

# 1. Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# 2. 경로 / 기본 설정

In [ ]:
import os
import numpy as np
import pandas as pd
from collections import Counter

CHOICES = ["a", "b", "c", "d"]
SCORE_COLS = ["score_a", "score_b", "score_c", "score_d"]

DATA_ROOT = "/content/drive/MyDrive/data"

DEV_CSV = os.path.join(DATA_ROOT, "dev.csv")
TEST_CSV = os.path.join(DATA_ROOT, "test.csv")

V3_DIR = "/content/drive/MyDrive/Qwen3VL_VQA_V3_1536_OCR_CROP"

V3_DEV_CANDIDATES = [
    os.path.join(V3_DIR, "dev_predictions_epoch_2.csv"),
    os.path.join(V3_DIR, "dev_predictions_epoch_1.csv"),
]

V3_TEST_CANDIDATES = [
    os.path.join(V3_DIR, "test_predictions_with_logits_and_crop.csv"),
    os.path.join(V3_DIR, "test_predictions_with_logits.csv"),
    os.path.join(V3_DIR, "test_predictions.csv"),
]

FINAL2_DIR = "/content/drive/MyDrive/Qwen3VL_FINAL2_PERM_TTA"

DEV_PERM_CACHE = os.path.join(FINAL2_DIR, "dev_perm_cache.csv")
TEST_PERM_CACHE = os.path.join(FINAL2_DIR, "test_perm_cache.csv")

OUTPUT_DIR = "/content/drive/MyDrive/Qwen3VL_FINAL3_VOTE_MICROTUNE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def first_existing(paths):
    for p in paths:
        if os.path.isfile(p):
            return p
    return None

V3_DEV = first_existing(V3_DEV_CANDIDATES)
V3_TEST = first_existing(V3_TEST_CANDIDATES)

print("DEV CSV :", DEV_CSV)
print("TEST CSV:", TEST_CSV)
print("V3 DEV  :", V3_DEV)
print("V3 TEST :", V3_TEST)
print("DEV CACHE:", DEV_PERM_CACHE)
print("TEST CACHE:", TEST_PERM_CACHE)

assert os.path.isfile(DEV_CSV)
assert os.path.isfile(TEST_CSV)
assert V3_DEV is not None
assert V3_TEST is not None
assert os.path.isfile(DEV_PERM_CACHE)
assert os.path.isfile(TEST_PERM_CACHE)

print("✅ 필요한 파일 확인 완료")

# 3. 데이터 / logits / permutation cache 로드

In [ ]:
dev_df = pd.read_csv(DEV_CSV)
test_df = pd.read_csv(TEST_CSV)

v3_dev = pd.read_csv(V3_DEV)
v3_test = pd.read_csv(V3_TEST)

dev_perm = pd.read_csv(DEV_PERM_CACHE)
test_perm = pd.read_csv(TEST_PERM_CACHE)

for df in [dev_df, test_df, v3_dev, v3_test, dev_perm, test_perm]:
    df["id"] = df["id"].astype(str)

# 원본 CSV 순서로 정렬
v3_dev = dev_df[["id"]].merge(v3_dev, on="id", how="left")
v3_test = test_df[["id"]].merge(v3_test, on="id", how="left")

assert not v3_dev[SCORE_COLS].isna().any().any()
assert not v3_test[SCORE_COLS].isna().any().any()

print("Dev:", len(dev_df))
print("Test:", len(test_df))
print("Dev perm cache rows :", len(dev_perm))
print("Test perm cache rows:", len(test_perm))

# 4. V3 probability / margin 계산

In [ ]:
def logits_to_probs(df):
    x = df[SCORE_COLS].to_numpy(dtype=np.float64)
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / ex.sum(axis=1, keepdims=True)

def get_features(p):
    order = np.sort(p, axis=1)[:, ::-1]
    idx = p.argmax(axis=1)

    return {
        "pred_idx": idx,
        "pred": np.array([CHOICES[i] for i in idx]),
        "confidence": order[:, 0],
        "margin": order[:, 0] - order[:, 1],
    }

p3_dev = logits_to_probs(v3_dev)
p3_test = logits_to_probs(v3_test)

f3_dev = get_features(p3_dev)
f3_test = get_features(p3_test)

print("V3 Test answer counts:")
print(pd.Series(f3_test["pred"]).value_counts().sort_index())

# 5. Permutation cache → sample별 3-view 정보

In [ ]:
def build_perm_map(cache):
    result = {}

    for rid, g in cache.groupby("id"):
        g = g.sort_values("perm_idx")

        # 3개 cyclic permutation이 모두 있어야 사용
        if len(g) < 3:
            continue

        p = g[
            ["prob_a", "prob_b", "prob_c", "prob_d"]
        ].to_numpy(dtype=np.float64)

        # NaN row는 제외
        if np.isnan(p).any():
            continue

        result[str(rid)] = {
            "views": p,
            "mean": p.mean(axis=0),
            "preds": p.argmax(axis=1),
        }

    return result

dev_perm_map = build_perm_map(dev_perm)
test_perm_map = build_perm_map(test_perm)

print("Complete Dev permutation items :", len(dev_perm_map))
print("Complete Test permutation items:", len(test_perm_map))

# 6. Dev 평가 함수

In [ ]:
DEV_ANSWER_COLS = [
    c for c in ["answer1", "answer2", "answer3", "answer4", "answer5"]
    if c in dev_df.columns
]

def get_valid_labels(row):
    labels = []

    for c in DEV_ANSWER_COLS:
        value = row[c]

        if pd.notna(value):
            value = str(value).strip().lower()

            if value in CHOICES:
                labels.append(value)

    return labels

def evaluate_dev(pred):
    consensus = []
    majority = []
    high_conf = []

    for i, (_, row) in enumerate(dev_df.iterrows()):
        labels = get_valid_labels(row)

        if not labels:
            continue

        p = pred[i]

        consensus.append(
            sum(x == p for x in labels) / len(labels)
        )

        counts = Counter(labels)
        mx = max(counts.values())

        winners = [
            k for k, v in counts.items()
            if v == mx
        ]

        if len(winners) == 1:
            hit = int(p == winners[0])
            majority.append(hit)

            if mx / len(labels) >= 0.60:
                high_conf.append(hit)

    return {
        "consensus": float(np.mean(consensus)),
        "majority": float(np.mean(majority)),
        "high_conf": float(np.mean(high_conf)),
    }

print("V3 Dev:", evaluate_dev(f3_dev["pred"]))

# 7. Vote-Switch 함수

규칙:

```text
base = V3

3개 permutation view의 argmax 투표
→ 최소 min_votes개가 같은 alternative 지지
→ alternative != V3 answer
→ V3 margin <= margin_cap
→ permutation mean에서
   P(alt) - P(V3 answer) >= delta
→ switch
```

FINAL2 성공 규칙은:

```text
min_votes = 2
delta = 0.08
margin_cap ≈ 0.50
```

입니다.

In [ ]:
def vote_switch_predictions(
    base_probs,
    base_features,
    source_df,
    perm_map,
    min_votes=2,
    delta=0.08,
    margin_cap=0.50,
):
    pred_idx = base_features["pred_idx"].copy()

    changed_meta = []

    for i in range(len(source_df)):
        if base_features["margin"][i] > margin_cap:
            continue

        rid = str(source_df.iloc[i]["id"])

        if rid not in perm_map:
            continue

        info = perm_map[rid]

        vote_counts = Counter(
            info["preds"].tolist()
        )

        alt_idx, n_votes = vote_counts.most_common(1)[0]
        base_idx = int(base_features["pred_idx"][i])

        if n_votes < min_votes:
            continue

        if alt_idx == base_idx:
            continue

        mean_p = info["mean"]

        prob_gap = (
            mean_p[alt_idx]
            - mean_p[base_idx]
        )

        if prob_gap < delta:
            continue

        pred_idx[i] = alt_idx

        changed_meta.append({
            "id": rid,
            "base": CHOICES[base_idx],
            "new": CHOICES[alt_idx],
            "votes": n_votes,
            "gap": prob_gap,
            "base_margin": base_features["margin"][i],
        })

    pred = np.array([
        CHOICES[i] for i in pred_idx
    ])

    return pred, pd.DataFrame(changed_meta)

# 8. 규칙 Grid Search — Dev + Test 변경 수 동시 계산

In [ ]:
MIN_VOTES_GRID = [2, 3]
DELTA_GRID = [0.00, 0.02, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10, 0.12, 0.15, 0.20]
MARGIN_CAP_GRID = [0.20, 0.30, 0.40, 0.50]

rows = []

for min_votes in MIN_VOTES_GRID:
    for delta in DELTA_GRID:
        for margin_cap in MARGIN_CAP_GRID:
            dev_pred, dev_changed = vote_switch_predictions(
                p3_dev,
                f3_dev,
                dev_df,
                dev_perm_map,
                min_votes=min_votes,
                delta=delta,
                margin_cap=margin_cap,
            )

            metrics = evaluate_dev(dev_pred)

            test_pred, test_changed = vote_switch_predictions(
                p3_test,
                f3_test,
                test_df,
                test_perm_map,
                min_votes=min_votes,
                delta=delta,
                margin_cap=margin_cap,
            )

            rows.append({
                "min_votes": min_votes,
                "delta": delta,
                "margin_cap": margin_cap,
                "dev_changed_n": len(dev_changed),
                "dev_changed_rate": len(dev_changed) / len(dev_df),
                "test_changed_n": len(test_changed),
                "test_changed_rate": len(test_changed) / len(test_df),
                **metrics,
            })

grid = pd.DataFrame(rows)

grid.to_csv(
    os.path.join(OUTPUT_DIR, "all_rule_candidates.csv"),
    index=False,
    encoding="utf-8-sig",
)

print("Top Dev candidates:")
display(
    grid.sort_values(
        ["consensus", "high_conf", "majority"],
        ascending=False,
    ).head(30)
)

# 9. 현재 최고 0.96127 규칙 재현

In [ ]:
ANCHOR = {
    "min_votes": 2,
    "delta": 0.08,
    "margin_cap": 0.50,
}

anchor_pred, anchor_changes = vote_switch_predictions(
    p3_test,
    f3_test,
    test_df,
    test_perm_map,
    **ANCHOR,
)

print("Anchor changed:", len(anchor_changes))
display(anchor_changes.head(20))

anchor_submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": anchor_pred,
})

anchor_path = os.path.join(
    OUTPUT_DIR,
    "00_anchor_delta008.csv",
)

anchor_submission.to_csv(
    anchor_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", anchor_path)

# 10. 수동 Micro-Tune 후보

현재 성공점 `delta=0.08`의 바로 양옆만 우선 만듭니다.

In [ ]:
MANUAL_RULES = {
    "01_looser_delta006.csv": {
        "min_votes": 2,
        "delta": 0.06,
        "margin_cap": 0.50,
    },
    "02_tighter_delta010.csv": {
        "min_votes": 2,
        "delta": 0.10,
        "margin_cap": 0.50,
    },
    "03_tighter_delta012.csv": {
        "min_votes": 2,
        "delta": 0.12,
        "margin_cap": 0.50,
    },
    "04_unanimous_3of3.csv": {
        "min_votes": 3,
        "delta": 0.00,
        "margin_cap": 0.50,
    },
    "05_unanimous_delta005.csv": {
        "min_votes": 3,
        "delta": 0.05,
        "margin_cap": 0.50,
    },
}

manual_summary = []

for filename, rule in MANUAL_RULES.items():
    pred, changes = vote_switch_predictions(
        p3_test,
        f3_test,
        test_df,
        test_perm_map,
        **rule,
    )

    sub = pd.DataFrame({
        "id": test_df["id"],
        "answer": pred,
    })

    path = os.path.join(
        OUTPUT_DIR,
        filename,
    )

    sub.to_csv(
        path,
        index=False,
        encoding="utf-8-sig",
    )

    manual_summary.append({
        "file": filename,
        **rule,
        "changed_n": len(changes),
        "changed_rate": len(changes) / len(test_df),
    })

manual_summary = pd.DataFrame(manual_summary)
display(manual_summary)

# 11. Dev-best Safe 후보 자동 생성

Public에 맞춰 과하게 탐색하지 않기 위해 Test 변경 개수에 제한을 둡니다.

현재 성공 규칙이 35개 변경이었으므로:

```text
Safe       : 10~60개 변경
Ultra-safe : 10~40개 변경
```

범위 안에서 Dev consensus가 높은 규칙을 고릅니다.

In [ ]:
def choose_safe_candidate(min_changes, max_changes):
    x = grid[
        (grid["test_changed_n"] >= min_changes)
        & (grid["test_changed_n"] <= max_changes)
    ].copy()

    if len(x) == 0:
        raise ValueError("조건에 맞는 candidate가 없습니다.")

    # Anchor와 지나치게 멀어지는 것을 막기 위한 tie-break
    x["anchor_distance"] = (
        abs(x["delta"] - 0.08)
        + 0.2 * abs(x["margin_cap"] - 0.50)
        + 0.05 * abs(x["min_votes"] - 2)
    )

    x = x.sort_values(
        [
            "consensus",
            "high_conf",
            "majority",
            "anchor_distance",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )

    return x.iloc[0]


best_safe = choose_safe_candidate(
    10,
    60,
)

best_ultra = choose_safe_candidate(
    10,
    40,
)

display(
    pd.DataFrame([
        {"name":"safe", **best_safe.to_dict()},
        {"name":"ultra_safe", **best_ultra.to_dict()},
    ])
)

# 12. Dev-best Safe submission 저장

In [ ]:
AUTO_RULES = {
    "06_dev_best_safe.csv": {
        "min_votes": int(best_safe["min_votes"]),
        "delta": float(best_safe["delta"]),
        "margin_cap": float(best_safe["margin_cap"]),
    },
    "07_dev_best_ultra_safe.csv": {
        "min_votes": int(best_ultra["min_votes"]),
        "delta": float(best_ultra["delta"]),
        "margin_cap": float(best_ultra["margin_cap"]),
    },
}

auto_summary = []

for filename, rule in AUTO_RULES.items():
    pred, changes = vote_switch_predictions(
        p3_test,
        f3_test,
        test_df,
        test_perm_map,
        **rule,
    )

    sub = pd.DataFrame({
        "id": test_df["id"],
        "answer": pred,
    })

    path = os.path.join(
        OUTPUT_DIR,
        filename,
    )

    sub.to_csv(
        path,
        index=False,
        encoding="utf-8-sig",
    )

    auto_summary.append({
        "file": filename,
        **rule,
        "changed_n": len(changes),
        "changed_rate": len(changes) / len(test_df),
    })

auto_summary = pd.DataFrame(auto_summary)

display(auto_summary)

# 13. 최종 후보 비교

In [ ]:
final_summary = pd.concat(
    [
        manual_summary,
        auto_summary,
    ],
    ignore_index=True,
)

# Anchor도 추가
anchor_row = pd.DataFrame([{
    "file": "00_anchor_delta008.csv",
    **ANCHOR,
    "changed_n": len(anchor_changes),
    "changed_rate": len(anchor_changes) / len(test_df),
}])

final_summary = pd.concat(
    [anchor_row, final_summary],
    ignore_index=True,
)

final_summary = final_summary.sort_values(
    "changed_n"
).reset_index(drop=True)

display(final_summary)

final_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final3_candidate_summary.csv",
    ),
    index=False,
    encoding="utf-8-sig",
)

# 14. 제출 순서 추천

현재 기준점:

```text
00_anchor_delta008.csv
= 기존 04_perm_vote_switch와 같은 규칙
= Public 0.96127
```

따라서 `00`은 다시 제출할 필요 없습니다.

## 제출 여유 1회

```text
02_tighter_delta010.csv
```

현재 최고 규칙보다 조금 보수적으로 틀린 switch를 줄이는 방향입니다.

## 2회

```text
01_looser_delta006.csv
```

35개보다 약간 더 많은 문제를 고치는 방향을 확인합니다.

## 3회

```text
04_unanimous_3of3.csv
```

세 permutation이 완전히 합의한 문제만 바꾸는 다른 성격의 후보입니다.

## 그 이상

`06_dev_best_safe.csv`, `07_dev_best_ultra_safe.csv`를 확인합니다.

---

### 제출 전 마지막 체크

`final3_candidate_summary.csv`에서 변경 개수를 확인하세요.

현재 성공 규칙은 **35개 변경**입니다.

대략:

```text
20~30개 : 더 보수적
30~45개 : 현재 성공 영역
45~60개 : 조금 공격적
60개 이상: 우선순위 낮음
```

이번 FINAL3는 새 모델을 건드리지 않고,
**이미 Public에서 효과가 확인된 Vote-Switch의 경계만 미세 조정**하는 마지막 단계입니다.